# Теория к Лабораторной работе №3
## «Очередь задач: итераторы и генераторы»

> **Цель:** Научиться реализовывать пользовательские коллекции и ленивую обработку задач.

### Что нужно реализовать
Класс `TaskQueue` — очередь задач с:
- **протоколом итерации** (`__iter__`, `__next__`)
- поддержкой **повторного обхода**
- **ленивыми фильтрами** по статусу и приоритету
- корректной работой с большими объёмами (нет избыточного хранения)
- совместимостью со стандартными конструкциями Python: `for`, `list()`, `sum()`


---
## 1. Iterable vs Iterator — в чём разница

| Понятие | Протокол | Что умеет |
|---------|----------|----------|
| **Iterable** | `__iter__()` | возвращает Iterator |
| **Iterator** | `__iter__()` + `__next__()` | выдаёт элементы по одному |

**Ключевое:** Iterator является Iterable (его `__iter__` возвращает `self`), но не наоборот.

```
for x in obj:
    ...
```
разворачивается в:
```python
it = iter(obj)          # вызывает obj.__iter__()
while True:
    try:
        x = next(it)    # вызывает it.__next__()
    except StopIteration:
        break
```

In [ ]:
# Разбираем for-loop вручную
data = [10, 20, 30]

it = iter(data)        # list.__iter__() → list_iterator
print(type(it))        # <class 'list_iterator'>

print(next(it))        # 10
print(next(it))        # 20
print(next(it))        # 30

try:
    print(next(it))    # StopIteration!
except StopIteration:
    print("итерация завершена")

# Список — Iterable, его итератор — Iterator:
print(iter(data) is data)   # False — список не является своим итератором
print(iter(it) is it)       # True  — итератор является своим итератором

<class 'list_iterator'>
10
20
30
итерация завершена
False
True


In [ ]:
# Проверка через isinstance
from collections.abc import Iterable, Iterator

lst = [1, 2, 3]
it  = iter(lst)

print(isinstance(lst, Iterable))  # True
print(isinstance(lst, Iterator))  # False — список не Iterator

print(isinstance(it, Iterable))   # True  — iterator тоже Iterable
print(isinstance(it, Iterator))   # True

# Строки, словари, генераторы — тоже Iterable:
print(isinstance("hello", Iterable))           # True
print(isinstance({"a": 1}, Iterable))          # True
print(isinstance((x for x in range(3)), Iterator))  # True

True
False
True
True
True
True
True


---
## 2. Протокол итерации: `__iter__` и `__next__`

In [ ]:
# Пример 1: Iterable, возвращающий отдельный Iterator
class CountDown:
    """Iterable: обратный отсчёт от n до 1. Повторяемый."""

    def __init__(self, start: int):
        self._start = start

    def __iter__(self):
        return CountDownIterator(self._start)  # новый объект каждый раз!


class CountDownIterator:
    """Iterator для CountDown."""

    def __init__(self, start: int):
        self._current = start

    def __iter__(self):
        return self   # Iterator возвращает self

    def __next__(self):
        if self._current <= 0:
            raise StopIteration
        value = self._current
        self._current -= 1
        return value


cd = CountDown(3)
print(list(cd))  # [3, 2, 1]
print(list(cd))  # [3, 2, 1] - повторный обход работает!

cd1 = CountDownIterator(3)
print(list(cd1))  # [3, 2, 1]
print(list(cd1))  # :)

[3, 2, 1]
[3, 2, 1]
[3, 2, 1]
[]


In [ ]:
# Пример 2: Класс совмещает Iterable и Iterator (одноразовый)
class CountDownOnce:
    """Iterator: обратный отсчёт. Одноразовый -- повтор не даст элементов."""

    def __init__(self, start: int):
        self._current = start

    def __iter__(self):
        return self   # себя же -- поэтому одноразовый!

    def __next__(self):
        if self._current <= 0:
            raise StopIteration
        value = self._current
        self._current -= 1
        return value


cd = CountDownOnce(3)
print(list(cd))  # [3, 2, 1]
print(list(cd))  # :) -- состояние не сбросилось

[3, 2, 1]
[]


---
## 3. `StopIteration` и безопасный выход

In [ ]:
# StopIteration — единственный правильный способ завершить итерацию
# next() со значением по умолчанию — безопасный вариант

it = iter([1, 2])
print(next(it, None))   # 1
print(next(it, None))   # 2
print(next(it, None))   # None  ← не StopIteration, а дефолт
print(next(it, -1))     # -1

# Внутри генератора return = StopIteration:
def finite():
    yield 1
    yield 2
    return   # эквивалент raise StopIteration

gen = finite()
print(next(gen))   # 1
print(next(gen))   # 2

try:
    next(gen)       # StopIteration
except StopIteration as e:
    print(f"Генератор завершён, значение return: {e.value}")

1
2
None
-1
1
2
Генератор завершён, значение return: None


---
## 4. Одноразовые и повторяемые итераторы

| Тип | `__iter__` возвращает | Повторный обход |
|-----|-----------------------|-----------------|
| **Iterable** (список, очередь) | новый Iterator | ✅ Да |
| **Iterator** (сам себя) | `self` | ❌ Нет |
| **Генератор** | `self` | ❌ Нет |

**Правило для коллекций:** коллекция должна быть **Iterable**, возвращающей **новый** Iterator при каждом вызове `__iter__`. Иначе `for ... for ...` на одной коллекции сломается.

In [1]:
# Демонстрация проблемы одноразового итератора
def task_generator(n):
    for i in range(n):
        yield f"task-{i}"

# Генератор — одноразовый Iterator
gen = task_generator(3)

# Вложенные циклы работают некорректно:
for outer in gen:
    for inner in gen:   # inner и outer берут из ОДНОГО итератора!
        print(f"outer={outer}, inner={inner}")
    break

print("---")

# Коллекция как Iterable — корректна:
class TaskBag:
    def __init__(self, tasks):
        self._tasks = list(tasks)

    def __iter__(self):
        return iter(self._tasks)  # новый list_iterator каждый раз

bag = TaskBag(["task-0", "task-1", "task-2"])

print("Первый проход:",  list(bag))
print("Второй проход:", list(bag))  # работает!

outer=task-0, inner=task-1
outer=task-0, inner=task-2
---
Первый проход: ['task-0', 'task-1', 'task-2']
Второй проход: ['task-0', 'task-1', 'task-2']


---
## 5. Генераторные функции и `yield`

**Генераторная функция** — функция, содержащая `yield`. При вызове возвращает **объект-генератор** (не результат).

Генератор:
- приостанавливается на `yield`, возвращает значение
- возобновляется при следующем `next()`
- помнит своё локальное состояние между вызовами

In [ ]:
# Базовый генератор
def fibonacci():
    """Бесконечная последовательность Фибоначчи."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

gen = fibonacci()
print([next(gen) for _ in range(10)])  # первые 10 чисел Фибоначчи

# Каждый вызов fibonacci() → новый независимый генератор:
gen1 = fibonacci()
gen2 = fibonacci()
print(next(gen1), next(gen1))   # 0 1
print(next(gen2))               # 0  ← независим от gen1

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
0 1
0


In [ ]:
# Генератор задач: вычисляет по одной, не загружает всё в память
from dataclasses import dataclass
from typing import Any

@dataclass
class Task:
    id: str
    description: str
    priority: int
    status: str = "new"


def generate_tasks(count: int):
    """Лениво генерирует задачи. В памяти одновременно — одна задача."""
    for i in range(count):
        yield Task(
            id=f"task-{i:04d}",
            description=f"Задача #{i}",
            priority=(i % 5) + 1,
        )

# Берём только первые 3 без генерации всех 1_000_000:
for task in generate_tasks(1_000_000):
    print(task)
    if task.priority == 3:
        break

Task(id='task-0000', description='Задача #0', priority=1, status='new')
Task(id='task-0001', description='Задача #1', priority=2, status='new')
Task(id='task-0002', description='Задача #2', priority=3, status='new')


---
## 6. `send`, `throw`, `close` — двусторонние генераторы

Генераторы поддерживают не только отдачу данных (`yield`), но и **приём** данных (`send`).

In [ ]:
# send(): отправить значение в генератор
def accumulator():
    """Генератор-аккумулятор: принимает числа, выдаёт накопленную сумму."""
    total = 0
    while True:
        print(total)
        value1 = yield total   # yield отдаёт total, получает следующее значение
        value2 = yield
        yield total + 100  # yield отдаёт total, получает следующее значение

        print(value1, value2, total)
        print("____")
        if value1 is None:
            break
        total += value1

gen = accumulator()
print(next(gen))           # «раскрутка» — дойти до первого yield
print("started")
print("out:", gen.send(10), gen.send(11))  # → 10
print(next(gen))
print("out:", gen.send(5))   # → 15
print("out:", gen.send(3))   # → 18
gen.close()          # завершить генератор


0
0
started
out: None 100
10 11 0
____
10
10
out: None
out: 110


In [ ]:
# throw(): бросить исключение в генератор
def safe_processor():
    while True:
        try:
            task = yield
            print(f"Обрабатываем: {task.id}")
        except ValueError as e:
            print(f"Пропускаем задачу: {e}")
        except GeneratorExit:
            print("Генератор завершён")
            return

proc = safe_processor()
next(proc)
proc.send(Task("t-1", "Тест", 3))
proc.throw(ValueError, "плохие данные")
proc.send(Task("t-2", "Тест 2", 4))
proc.close()

Обрабатываем: t-1
Пропускаем задачу: плохие данные
Обрабатываем: t-2
Генератор завершён


C:\Users\PC\AppData\Local\Temp\ipykernel_69036\2691353386.py:16: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  proc.throw(ValueError, "плохие данные")


---
## 9. `yield from` — делегирование итерации

In [ ]:
# yield from: делегировать итерацию вложенному iterable

# Без yield from:
def chain_manual(*iterables):
    for it in iterables:
        for item in it:
            yield item

# С yield from (эквивалент, но эффективнее и поддерживает send/throw):
def chain(*iterables):
    for it in iterables:
        yield from it   # делегирует итерацию напрямую

sources = [
    [Task("a-1", "A1", 1), Task("a-2", "A2", 2)],
    [Task("b-1", "B1", 3)],
    [Task("c-1", "C1", 4), Task("c-2", "C2", 5)],
    [Task("c-1", "C1", 4), Task("c-2", "C2", 5), Task("c-2", "C2", 5)],
]

all_tasks = list(chain(*sources))
print(f"Всего задач: {len(all_tasks)}")
for t in all_tasks:
    print(f"  {t.id}  priority={t.priority}")

Всего задач: 8
  a-1  priority=1
  a-2  priority=2
  b-1  priority=3
  c-1  priority=4
  c-2  priority=5
  c-1  priority=4
  c-2  priority=5
  c-2  priority=5


In [ ]:
# yield from со значением return (PEP 380)
def sub_generator():
    yield 1
    yield 2
    return "результат sub"   # значение, передаваемое через StopIteration.value

def delegating_generator():
    result = yield from sub_generator()  # result получает значение return
    print(f"sub завершился с: {result!r}")
    yield 99

gen = delegating_generator()
print(next(gen))   # 1
print(next(gen))   # 2
print(next(gen))   # выводит "sub завершился с: 'результат sub'", возвращает 99

1
2
sub завершился с: 'результат sub'
99
